# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema (JSON-LD) accessible via:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, record sets are accessed by their `@id`. We print available record sets along with field and column `@id` information.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets

print("Available Record Sets (referenced by @id):")

for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  Field @id: {field['@id']} (name: {field.get('name', field['@id'])})")
            if 'column' in field:
                column = field['column']
                print(f"    Column @id: {column['@id']} (label: {column.get('label', column['@id'])})")
    print()

# Example: Show records from the first record set
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"Sample records from Record Set {example_record_set_id}:")
    for x in dataset.records(record_set=example_record_set_id):
        print(x)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use record set and field `@id`s as referenced above.

In [ ]:
# Collect all record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for Record Set {record_set_id}: Columns -> {df.columns.tolist()}")
    print(df.head(3))

# Choose a record set to analyze further (first one for demonstration)
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id:
    print(f"Selected Record Set for EDA: {primary_record_set_id}")
    print(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All fields are referenced by their `@id` per Croissant schema.

In [ ]:
# Example EDA steps: filter, normalize, and group
df = dataframes[primary_record_set_id]

# Let us select a numeric field by @id (update as needed based on available fields; here we use the first field that is numeric)
numeric_field_id = None
for rs in record_sets:
    if rs['@id'] == primary_record_set_id:
        if 'fields' in rs:
            for field in rs['fields']:
                # Attempt to select a Float or Integer type
                if field.get('dataType', '').endswith('Float') or field.get('dataType', '').endswith('Integer'):
                    numeric_field_id = field['@id']
                    break

if numeric_field_id and numeric_field_id in df:
    # Filter: e.g. select records with numeric_field > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (e.g. choose a categorical field @id)
    group_field_id = None
    for field in rs['fields']:
        if field['@id'] != numeric_field_id and not (field.get('dataType', '').endswith('Float') or field.get('dataType', '').endswith('Integer')):
            group_field_id = field['@id']
            break

    if group_field_id and group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

All visualization axes and labels should refer to fields by `@id`.

In [ ]:
if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show boxplot
    if group_field_id and group_field_id in df:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Through this notebook, we've demonstrated:
- How to load and interpret a FAIR^2 dataset with Croissant schema using `mlcroissant`.
- Access and reference all dataset elements by their `@id`, ensuring clear mappings for fields, record sets, and columns.
- Basic exploratory analysis and visualization on the dataset.

Given the dataset's limited size and focused clinical scope, further statistical or machine learning modeling is not recommended without expansion and validation. Analytical steps illustrated here serve as a case-based demonstration for rare disease research and reproducibility.